In [ ]:
!pip install scikit-learn

In [ ]:
pip install opencv-python tqdm

In [ ]:
# Frame Extraction (simplified)

import os
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split

def extract_frames(video_path, output_folder, frame_skip=5):
    os.makedirs(output_folder, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_skip == 0:
            frame = cv2.resize(frame, (112, 112))
            cv2.imwrite(os.path.join(output_folder, f"frame_{count:03d}.jpg"), frame)
        count += 1
    cap.release()

def prepare_dataset(src_dir, out_dir="dataset", test_ratio=0.2):
    categories = [("Violence", "violent"), ("NonViolence", "non_violent")]
    video_list = []

    for cat, label in categories:
        full_dir = os.path.join(src_dir, cat)
        for f in os.listdir(full_dir):
            if f.endswith(".mp4"):
                video_list.append((os.path.join(full_dir, f), label))

    from sklearn.model_selection import train_test_split
    train, test = train_test_split(video_list, test_size=test_ratio, stratify=[lbl for _, lbl in video_list])

    for split, data in [("train", train), ("test", test)]:
        for path, label in tqdm(data, desc=split):
            name = os.path.splitext(os.path.basename(path))[0]
            out_path = os.path.join(out_dir, split, label, name)
            extract_frames(path, out_path)

if __name__ == "__main__":
    prepare_dataset("/mnt/e/Placement_Project/Dataset/Real Life Violence Dataset")

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

class ClipDataset(Dataset):
    def __init__(self, root_dir, clip_len=16):
        self.samples = []
        self.clip_len = clip_len
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

        for label in ['violent', 'non_violent']:
            class_dir = os.path.join(root_dir, label)
            for video_folder in os.listdir(class_dir):
                frame_dir = os.path.join(class_dir, video_folder)
                frame_files = sorted(os.listdir(frame_dir))
                if len(frame_files) >= clip_len:
                    self.samples.append((frame_dir, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_dir, label = self.samples[idx]
        frames = sorted(os.listdir(frame_dir))[:self.clip_len]
        clip = [self.transform(Image.open(os.path.join(frame_dir, f)).convert("RGB")) for f in frames]
        clip = torch.stack(clip)
        return clip, torch.tensor(1 if label == "violent" else 0)

In [ ]:
# models/cnn_lstm_model.py

import torch
import torch.nn as nn
import torchvision.models as models

class CNNLSTM(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=1, bidirectional=False):
        super(CNNLSTM, self).__init__()
        
        # Use pretrained ResNet18 and remove the final classification layer
        self.cnn = models.resnet18(pretrained=True)
        self.cnn.fc = nn.Identity()  # remove the final FC layer to get features
        
        # LSTM to process sequence of frame features
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )
        
        direction_factor = 2 if bidirectional else 1
        self.classifier = nn.Linear(hidden_dim * direction_factor, 1)

    def forward(self, x):  # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape
        
        # Merge batch and time to process through CNN
        x = x.view(B * T, C, H, W)
        features = self.cnn(x)  # [B*T, 512]
        
        # Reshape back to sequence format
        features = features.view(B, T, -1)  # [B, T, 512]
        
        # LSTM
        lstm_out, _ = self.lstm(features)  # [B, T, H]
        out = lstm_out[:, -1, :]  # Take last time step output
        
        # Classification
        out = self.classifier(out)  # [B, 1]
        return out.squeeze(1)


In [ ]:
import torch
from torch.utils.data import DataLoader
from models.cnn_lstm_model import CNNLSTM
import torch.nn as nn
import torch.optim as optim


def train_model(epochs=10, batch_size=4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CNNLSTM().to(device)
    train_set = ClipDataset("dataset/train")
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0
        for clips, labels in train_loader:
            clips, labels = clips.to(device), labels.float().to(device)
            outputs = model(clips)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total_loss += loss.item()

        acc = correct / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {total_loss:.4f} Accuracy: {acc:.4f}")

    torch.save(model.state_dict(), "cnn_lstm_model.pth")


if __name__ == "__main__":
    train_model()